# SPARC Example 22: Distance Uncertainty and Its Effect on Omega

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

SPARC distance uncertainties (e_distance_mpc) are published by Lelli et al. (2016).
Since R_kpc = R_arcsec * D_Mpc, distance uncertainty propagates into
the radius scale and therefore omega.

This example quantifies how much omega changes with ±1σ distance.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

g = next(g for g in corpus['galaxies'] if g['galaxy'] == 'DDO161')
d = g['data']
D      = g['distance_mpc']
e_D    = g.get('e_distance_mpc', D * 0.1)

R    = np.array([p['Rad']  for p in d])
Vobs = np.array([p['Vobs'] for p in d])

def compute_omega(R, Vobs):
    R1, V1 = R[0],  Vobs[0]
    R2, V2 = R[-1], Vobs[-1]
    if R1<=0 or R2<=0:
        return None
    return (V2/R2 - V1/R1) * (R1/R2)**1.5

omega_nominal = compute_omega(R, Vobs)

# Scale radii by distance
omega_plus  = compute_omega(R * (D + e_D) / D, Vobs)
omega_minus = compute_omega(R * (D - e_D) / D, Vobs)

print(f"DDO161:")
print(f"  Distance:      {D:.2f} ± {e_D:.2f} Mpc")
print(f"  Omega nominal: {omega_nominal:.3f} rad/Gyr")
print(f"  Omega D+1σ:    {omega_plus:.3f} rad/Gyr")
print(f"  Omega D-1σ:    {omega_minus:.3f} rad/Gyr")
print(f"  Delta omega:   ±{(omega_plus - omega_minus)/2:.3f} rad/Gyr")
print(f"  Fractional:    {abs(omega_plus-omega_minus)/(2*omega_nominal)*100:.1f}%")

DDO161:
  Distance:      7.50 ± 2.25 Mpc
  Omega nominal: -0.207 rad/Gyr
  Omega D+1σ:    -0.159 rad/Gyr
  Omega D-1σ:    -0.295 rad/Gyr
  Delta omega:   ±0.068 rad/Gyr
  Fractional:    -33.0%
